<h1 align="center">Laboratorio 4</h1>

## Información

**Integrantes:**

| Name              | Institution ID | GitHub User |
| ----------------- | -------------- | ----------- |
| Josué Say         | 22801          | JosueSay    |
| Carlos Valladares | 221164         | vgcarlol    |

- [Repositorio](https://github.com/JosueSay/intro-to-computer-vision/tree/main/labs/lab4)

## Preparación de entorno

In [ ]:
# %pip install -r requirements.txt
# jupyter nbconvert lab4.ipynb --to html

## Librerías y Configuración

In [ ]:
import cv2, math, os, time
import numpy as np
import matplotlib.pyplot as plt
from tabulate import tabulate

## Task 1

El objetivo es evaluar la comprensión de la geometría proyectiva y la manipulación algebraica de coordenadas homogéneas. Entonces, responda las siguientes preguntas demostrando el desarrollo matemático. No se aceptan respuestas puramente textuales sin respaldo algebraico.

### Inciso 1

Una homografía $H$ es una matriz de $3 \times 3$. Explique matemáticamente por qué, aunque tiene 9 elementos, solo posee 8 grados de libertad (GDL)

**Respuesta:**

Teniendo la homografía:

$$
H=
\begin{bmatrix}
h_{11} & h_{12} & h_{13} \\
h_{21} & h_{22} & h_{23} \\
h_{31} & h_{32} & h_{33}
\end{bmatrix},
\quad
\mathbf{x}=
\begin{bmatrix}
x \\
y \\
1
\end{bmatrix},
\quad
\mathbf{x'} \sim H\mathbf{x}
$$


En coordenadas homogéneas como una equivalencia de escala:

$$
\mathbf{x}\sim \lambda\mathbf{x},\ \lambda\neq 0
$$

Entonces para cualquier $\alpha\neq 0$:

$$
\mathbf{x'}\sim H\mathbf{x}\ \Rightarrow\ \mathbf{x'}\sim (\alpha H)\mathbf{x}
$$

porque el resultado homogéneo final se normaliza dividiendo por la tercera coordenada. Es decir, (H) está definida hasta escala:

$$
H \sim \alpha H
$$

Esto introduce una ambigüedad de 1 parámetro (la escala global), así que:

$$
\text{GDL}(H)=9-1=8
$$

#### Inciso a

Adicionalmente, responda. Si tuviéramos una cámara que solo rota sobre su eje óptico (sin traslación ni cambio de perspectiva), ¿la matriz de transformación sigue teniendo 8 GDL o se reduce? Demuestre la estructura de dicha matriz simplificada.

**Respuesta:**

Si la cámara solo rota alrededor del eje óptico, en el plano imagen eso es una isometría 2D centrada en el punto principal.

> isometría = rotación + traslación (3 GDL). Se restringe a una sola rotación y el centro está fijado por intrínsecos.

En forma homogénea, una rotación 2D alrededor del origen es:

$$
R(\theta)=
\begin{bmatrix}
\cos\theta & -\sin\theta & 0\
\sin\theta & \cos\theta & 0\
0&0&1
\end{bmatrix}.
$$

Pero rotar alrededor del **punto principal** $c_x,c_y$:

$$
T(c_x,c_y)=
\begin{bmatrix}
1&0&c_x\
0&1&c_y\
0&0&1
\end{bmatrix},\quad
T(-c_x,-c_y)=
\begin{bmatrix}
1&0&-c_x\
0&1&-c_y\
0&0&1
\end{bmatrix}.
$$

Entonces la transformación en la imagen es:

$$
H_{\text{roll}} = T(c_x,c_y), R(\theta), T(-c_x,-c_y).
$$

Al expander queda la multiplicación directa:

$$
H_{\text{roll}}=
\begin{bmatrix}
\cos\theta & -\sin\theta & c_x - c_x\cos\theta + c_y\sin\theta\
\sin\theta & \cos\theta & c_y - c_x\sin\theta - c_y\cos\theta\
0&0&1
\end{bmatrix}.
$$


- Es afín/isométrica (tercera fila $[0\ 0\ 1]$ => sin componente proyectiva).
- Los parámetros libres aquí son solo $\theta$ (si $c_x,c_y$ se consideran parte de (K) y fijos).

Ya no son 8 GDL, se reduce. En el caso ideal “solo roll, sin traslación ni perspectiva”:

$$
\text{GDL} = 1\ (\theta).
$$

### Inciso 2

En el algoritmo DLT (Direct Linear Transform), convertimos el problema $x' = Hx$ en un sistema de la forma $Ah = 0.$

Explique por qué buscamos el vector singular asociado al menor valor singular de $A$ en lugar de simplemente invertir la matriz. ¿Qué representa geométricamente ese "menor valor singular" cuando los datos tienen ruido?

**Respuesta:**

DLT transforma:

$$
\mathbf{x'} \sim H\mathbf{x}
$$

a un sistema homogéneo:

$$
A\mathbf{h} = 0,
$$

donde $\mathbf{h}$ apila los 9 parámetros de $H$.

**No se puede “invertir” porque no es (Ax=b)**

No es un vector $b\neq 0$, es un sistema homogéneo. La solución trivial siempre existe:

$$
\mathbf{h}=0
$$

y no sirve. Además, $A$ típicamente es $2N\times 9$; no es cuadrada => **no es invertible**.

Por eso se impone la restricción:

$$
|\mathbf{h}|=1
$$

y se resuelve el problema de mínimos cuadrados:

$$
\min_{|\mathbf{h}|=1} |A\mathbf{h}|
$$

**¿Por qué el menor valor singular?**

Usando SVD:

$$
A = U\Sigma V^T
$$

Entonces:

$$
|A\mathbf{h}| = |U\Sigma V^T \mathbf{h}| = |\Sigma (V^T\mathbf{h})|
$$

porque $U$ es ortonormal y preserva norma.

Si definimos:

$$
\mathbf{y}=V^T\mathbf{h},\quad |\mathbf{y}|=|\mathbf{h}|=1
$$

entonces:

$$
|A\mathbf{h}|^2 = |\Sigma \mathbf{y}|^2
= \sum_{i} \sigma_i^2, y_i^2
$$

Para minimizar esto con $|\mathbf{y}|=1$, conviene poner toda la "energía" en el componente con $(\sigma_i)$ más pequeño, o sea el menor valor singular. Por eso:

$$
\mathbf{h} = \mathbf{v}_{\min}
$$

**Interpretación geométrica con ruido**

Con ruido, ya no existe $\mathbf{h}$ tal que $A\mathbf{h}=0$. El menor valor singular $\sigma_{\min}$ mide qué tan cerca está el sistema de tener un nullspace perfecto de dimensión 1 para esa solución o equivalentemente, el residuo mínimo alcanzable:

$$
\min_{|\mathbf{h}|=1}|A\mathbf{h}| = \sigma_{\min}
$$

### Inciso 3

Si usted selecciona 4 puntos para calcular $H$, pero 3 de ellos son colineales (están en la misma línea recta), el algoritmo fallará.

Explique algebraicamente qué le sucede a la matriz $A$ del sistema DLT en este caso y por qué no tiene solución única.

**Respuesta:**

En DLT para homografía, cada correspondencia aporta 2 ecuaciones => con 4 puntos:

$$
A\in \mathbb{R}^{8\times 9}
$$

Para una solución única (hasta escala), queremos que el espacio nulo sea 1D:

$$
\dim(\mathcal{N}(A)) = 1
\ \Longleftrightarrow
\operatorname{rank}(A)=8
$$

**Colinealidad => pérdida de rango**

Si 3 puntos $\mathbf{x}_1,\mathbf{x}_2,\mathbf{x}_3$ son colineales, existe una recta $\mathbf{l}$ tal que:

$$
\mathbf{l}^T \mathbf{x}_i = 0,\quad i=1,2,3
$$

Esto induce dependencias lineales en las ecuaciones que se apilan en $A$; varias filas construidas desde esos puntos se vuelven linealmente dependientes, porque la geometría no excita todos los parámetros de $H$.

Resultado:

$$
\operatorname{rank}(A) < 8
\quad \Rightarrow\quad
\dim(\mathcal{N}(A)) = 9-\operatorname{rank}(A) > 1
$$

O sea, hay infinitas soluciones $\mathbf{h}$ (no solo una dirección), por lo que $H$ no queda determinada de manera única.

## Task 2

El objetivo de esta parte es implementar el pipeline de alineación sin depender de cajas negras. Por ello considere que no deben de usar cv2.findHomography o cv2.RANSAC. Además para este laboratorio necesitará crear su propio dataset, por ello tome 3 fotografías propias de una escena planar (i.e. una pancarta en una pared, un cuadro, o una fachada de edificio lejana) con ángulos y perspectivas drásticamente diferentes. Con esto realice:

### Inciso 1 - Detección y Macheo

#### Inciso a

Utilice SIFT u ORB (permitido usar OpenCV aquí) para detectar puntos de interés y descriptores.

#### Inciso b

Realice un emparejamiento de fuerza bruta (Brute-Force Matcher).

#### Inciso c

Requisito: Visualice los matches antes de filtrar. Debe verse una cantidad considerable de ruido/errores.

### Inciso 2 - Algoritmo DLT

#### Inciso a

Escriba una función calcular_homografia_dlt(puntos_src, puntos_dst) que reciba exactamente 4 pares de puntos.

#### Inciso b

Debe construir la matriz $A$ de tamaño $8 \times 9$.

#### Inciso c

Debe resolver el sistema usando SVD (numpy.linalg.svd).

#### Inciso d

Nota: Debe normalizar los puntos antes de entrar al DLT (restar la media y dividir por la desviación estándar) para estabilidad numérica, y des-normalizar la matriz $H$ resultante.

### Inciso 3 - RANSAC Manual

#### Inciso a

Implemente la función ransac_homografia(matches, umbral, prob_exito).

#### Inciso b

Cálculo de N: Su código debe calcular dinámicamente el número de iteraciones $N$ basado en la fórmula de probabilidad vista en clase. No "hardcodee" el número 1000.

#### Inciso c - Bucle

##### Inciso i

Seleccione 4 matches aleatorios.

##### Inciso ii

Llame a su función DLT.

##### Inciso iii

Proyecte todos los puntos fuente usando H_test.

##### Inciso iv

Calcule el error de reproyección (distancia Euclidiana) y cuente los inliers.

#### Inciso d

Refinamiento: Una vez encontrado el mejor conjunto de inliers, recalcule $H$ final usando todos los inliers (no solo los 4 iniciales) mediante SVD.

> La salida esperada es una imagen “stitched” (panorama) mostrando la alineación correcta.

## Task 3

En esta parte lo que se busca es evaluar el criterio profesional ante situaciones adversas y trade-offs de diseño. Para ello realice y responda lo siguiente

### Inciso 1

Ejecute su algoritmo RANSAC variando el parámetro de umbral de error (threshold) en pixeles (ej. 1px, 5px, 20px)

#### Inciso a

Genere una gráfica: Eje X = Umbral, Eje Y = Número de Inliers encontrados.

#### Inciso b

Discusión: Como ingeniero, ¿qué riesgo corre si establece un umbral demasiado estricto (ej. 0.5px)? ¿Qué pasa con la matriz final si el umbral es muy laxo (ej. 50px)?

### Inciso 2

Imagine que usted es el Lead Computer Vision Engineer de una empresa de drones. Deben alinear imágenes térmicas de paneles solares tomadas desde el aire para detectar fallos.

#### Inciso a

Problema: El drone vuela a 50 metros de altura. El terreno no es perfectamente plano (hay colinas suaves), pero los paneles sí son planos.

#### Inciso b

Pregunta A: ¿Es válido usar una Homografía global para unir todo el mapa del terreno? ¿Por qué sí o por qué no?

#### Inciso c

Pregunta B: Su algoritmo RANSAC está tardando demasiado (3 segundos por frame) en la computadora a bordo del drone (Raspberry Pi). La telemetría indica que el 90% de los matches iniciales son outliers debido al pasto y árboles repetitivos.

##### Inciso i

Proponga una estrategia concreta para reducir el tiempo de ejecución sin cambiar el hardware. (Pista: Piense en la fórmula de N o en pre-filtrado geométrico)